In [1]:
import lxml.html
import requests
import time
import uuid
import pandas as pd

In [2]:
def make_request(url):
    user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    acc = 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
    headers = {
    'User-Agent': user_agent,
    'Accept': acc,
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
    }

    resp = requests.get(url, headers = headers)
    print(f"Scraped with status code {resp.status_code}")

    if resp.status_code == 429:
        time.sleep(2.5)
        resp = make_request(url)
    elif resp.status_code == 200:
        return resp

In [3]:
def scrape_judge(url):
    j_page = lxml.html.fromstring(make_request(url).text)
    r_d = {}

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//h3/text()"
    item_titles = j_page.xpath(xp1 + xp2)

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//p/text()"
    items = j_page.xpath(xp1 + xp2)

    for i, title in enumerate(item_titles):
        r_d[title.lower()] = items[i].replace("\t", "").strip().lower()

    return r_d

In [4]:
url = "https://state-law-research.org/state-justices/"

root = lxml.html.fromstring(make_request(url).text)

xp1 = "//section[@filter = 'judge']//"
xp2 = "a[contains(@href, 'judge')]//@href"
judge_links = root.xpath(xp1 + xp2)

xp1 = "//section[@filter = 'judge']//a[contains(@href, 'judge')]//"
xp2 = "div[@class = 'module--content module--content-post']"
xp3 = "//h2[@class = 'title']/text()"
judge_names = root.xpath(xp1 + xp2 + xp3)

xp1 = "//div[@class = 'about-icons']"
xp2 = "//div[@class = 'about-icon' and @data-type = 'state']"
xp3 = "//span[not(contains(@class, 'sr-only'))]/text()"
judge_states = root.xpath(xp1 + xp2 + xp3)

judge_pd = {
    "JID": [],
    "name": [],
    "state": [],
    "gender": [],
    "party": [],
    "race": [],
    "professional experience": [],
    "election type": [],
    "term start": [],
    "term end": [],
    "next election date": []
}

Scraped with status code 200


In [5]:
for i, name in enumerate(judge_names):
    judge_pd["name"].append(name)
    judge_pd["JID"].append(uuid.uuid4())
    judge_pd["state"].append(judge_states[i])

    print(f"Scraping webpage for judge {name}")
    judge_dic = scrape_judge(judge_links[i])

    if judge_dic == {}:
        break

    judge_fields = [
        "gender",
        "party",
        "race",
        "professional experience",
        "election type",
        "term start",
        "term end",
        "next election date"
    ]

    for item in judge_fields:

        if item not in judge_dic:
            data = pd.NA
        else:
            data = judge_dic[item]

        judge_pd[item].append(data)

Scraping webpage for judge Abigail LeGrow
Scraped with status code 200
Scraping webpage for judge Adam Tanenbaum
Scraped with status code 200
Scraping webpage for judge Aimee A. Oravec
Scraped with status code 200
Scraping webpage for judge Alisa Kelli Wise
Scraped with status code 200
Scraping webpage for judge Allison Riggs
Scraped with status code 200
Scraping webpage for judge Andrew J. McDonald
Scraped with status code 200
Scraping webpage for judge Andrew M. Horton
Scraped with status code 200
Scraping webpage for judge Andrew M. Mead
Scraped with status code 200
Scraping webpage for judge Andrew Pinson
Scraped with status code 200
Scraping webpage for judge Angela M. Eaves
Scraped with status code 200
Scraping webpage for judge Angela McCormick Bisig
Scraped with status code 200
Scraping webpage for judge Anita Earls
Scraped with status code 200
Scraping webpage for judge Ann A. Scott Timmer
Scraped with status code 200
Scraping webpage for judge Anna Blackburne-Rigsby
Scraped w

In [6]:
for key in judge_pd.keys():
    print(f"{key}: {len(judge_pd[key])}")

JID: 345
name: 345
state: 345
gender: 345
party: 345
race: 345
professional experience: 345
election type: 345
term start: 345
term end: 345
next election date: 345


In [7]:
pd.DataFrame(judge_pd)

,JID,name,state,gender,party,race,professional experience,election type,term start,term end,next election date
0,e6a23d74-7f1b-471e-a22e-ca0d5c8e3dd9,Abigail LeGrow,DE,female,unsure,white,"corporate, judicial","appointed, leg confirmed","may 3, 2023",2035,requires re-nomination and re-confirmation
1,fb162868-f96a-4a2b-b54c-7dd6e15db32f,Adam Tanenbaum,FL,male,republican,white,"government civil (non civil rights), judicial,...",appointed,"january 14, 2026","january 2, 2029",<NA>
2,c319376a-28be-4c09-a7de-aa7938e4d819,Aimee A. Oravec,AK,female,unsure,white,"corporate, misc. private practice",appointed,"january 31, 2025",2028,2028
3,4c17e11a-f27c-4bfd-bec4-cf020e3108fd,Alisa Kelli Wise,AL,female,republican,white,"corporate, misc. private practice, judicial, o...","elected, partisan",2011,"january 15, 2029","november 2, 2028"
4,237a9dee-7042-453c-b26a-f5cd7814773f,Allison Riggs,NC,female,democrat,white,"legal aid/non-govt civil rights, judicial",appointed,"september 11, 2023","december 31, 2024","november 7, 2024"
...,...,...,...,...,...,...,...,...,...,...,...
340,c7ffc225-2436-4919-8e99-903a9fed909b,"William J. Musseman, Jr.",OK,male,republican,white,"prosecutor, judicial",appointed,2022,unsure,unsure
341,c25d2def-c0f0-4f2d-833e-d71577956f59,William P. Robinson III,RI,male,republican,white,misc. private practice,"appointed, leg confirmed",2004,lifetime,n/a - appointed for life
342,6a7d4af0-ce6f-477c-91c7-9f206564c5c7,William R. Wooton,WV,male,democrat,white,"prosecutor, plaintiff-side civil private pract...","elected, nonpartisan","january 1, 2021",december 2032,tbd
343,6cf8c629-b4e6-4076-8f79-fc3b4cc0e4b4,"William W. Hood, III",CO,male,democrat,white,"prosecutor, corporate, plaintiff-side civil pr...","appointed, retention elected","january 13, 2014","january 11, 2027","november 3, 2026"
